### 定义网络环境参数：

In [2]:
import numpy as np

# 网格行数和列数
rows = 3
cols = 4

# 定义奖励矩阵，对应网格中每个状态的即时奖励
rewards = np.array([
    [-0.02, -0.02, -0.02, 1],
    [-0.02, -0.02, -0.02, -np.inf],  # 障碍物区域设为负无穷大表示不可达
    [-0.02, -0.02, -0.02, -0.02]
])

# 定义动作，上下左右，分别用 0,1,2,3 表示
actions = [0, 1, 2, 3]  # 0:上, 1:下, 2:左, 3:右
action_symbols = ["↑", "↓", "←", "→"]  # 用于展示策略

# 折扣因子
gamma = 0.9

# 价值函数初始化，初始时所有状态价值为 0
value_function = np.zeros((rows, cols))

### 定义状态转移函数：

In [3]:
def transition(state, action):
    """
    状态转移函数，根据当前状态和动作，返回下一个状态
    :param state: 当前状态 (row, col)
    :param action: 动作
    :return: 下一个状态 (new_row, new_col)
    """
    row, col = state
    if action == 0:  # 上
        new_row = max(row - 1, 0)
        new_col = col
    elif action == 1:  # 下
        new_row = min(row + 1, rows - 1)
        new_col = col
    elif action == 2:  # 左
        new_row = row
        new_col = max(col - 1, 0)
    else:  # 右
        new_row = row
        new_col = min(col + 1, cols - 1)

    # 检查是否是障碍区域（这里简单判断奖励为负无穷的情况，实际可根据需求调整）
    if rewards[new_row, new_col] == -np.inf:
        return (row, col)  # 留在原状态
    return (new_row, new_col)

### 价值迭代算法实现：

In [4]:
# 价值迭代迭代次数，相邻两次迭代的价值函数差值小于某个阈值（如 1e-6)时，可认为收敛
iterations = 100

for _ in range(iterations):
    new_value_function = np.copy(value_function)
    for i in range(rows):
        for j in range(cols):
            if rewards[i, j] == -np.inf:
                continue  # 障碍区域不更新
            q_values = []
            for action in actions:
                next_state = transition((i, j), action)
                next_row, next_col = next_state
                q_value = rewards[i, j] + gamma * value_function[next_row, next_col]
                q_values.append(q_value)
            new_value_function[i, j] = max(q_values)
    value_function = new_value_function

### 获取最优策略：

In [5]:
# 提取最优策略
optimal_policy = np.zeros((rows, cols), dtype=int)
for i in range(rows):
    for j in range(cols):
        if rewards[i, j] == -np.inf:
            optimal_policy[i, j] = -1  # 障碍区域标记
            continue
        q_values = []
        for action in actions:
            next_state = transition((i, j), action)
            next_row, next_col = next_state
            q_value = rewards[i, j] + gamma * value_function[next_row, next_col]
            q_values.append(q_value)
        optimal_policy[i, j] = np.argmax(q_values)

### 结果展示：

In [6]:
# 展示价值函数
print("价值函数:")
print(value_function)

# 找到并展示一条最优路径
def find_optimal_path(start_state, goal_state):
    """
    根据最优策略找到从起始状态到目标状态的路径
    :param start_state: 起始状态 (row, col)
    :param goal_state: 目标状态 (row, col)
    :return: 路径列表，每个元素是 (状态, 动作)
    """
    path = []
    current_state = start_state
    
    # 防止无限循环，设置最大步数
    max_steps = rows * cols
    steps = 0
    
    while current_state != goal_state and steps < max_steps:
        row, col = current_state
        action = optimal_policy[row, col]
        
        # 如果是障碍物，跳出循环
        if action == -1:
            break
            
        # 记录当前状态和动作
        path.append((current_state, action_symbols[action]))
        
        # 移动到下一个状态
        current_state = transition(current_state, action)
        steps += 1
    
    # 添加目标状态
    if current_state == goal_state:
        path.append((current_state, "目标"))
    
    return path

# 设置起始点和目标点
start_state = (1, 1)  # (2,2) 对应数组索引 (1,1)
goal_state = (0, 3)   # (1,4) 对应数组索引 (0,3)

# 找到最优路径
optimal_path = find_optimal_path(start_state, goal_state)

# 创建一个空的网格，用于显示最优路径
path_grid = np.full((rows, cols), '.', dtype=str)  # 使用点表示未访问的格子

# 标记障碍物
for i in range(rows):
    for j in range(cols):
        if rewards[i, j] == -np.inf:
            path_grid[i, j] = 'X'  # 障碍物

# 标记路径上的动作
for i in range(len(optimal_path) - 1):
    state, action = optimal_path[i]
    row, col = state
    path_grid[row, col] = action

# 标记起点和终点
if optimal_path:
    start_row, start_col = optimal_path[0][0]
    path_grid[start_row, start_col] = 'S'  # 起点
    
    end_row, end_col = optimal_path[-1][0]
    path_grid[end_row, end_col] = 'G'  # 终点

# 展示价值函数
print("价值函数:")
print(value_function)

# 展示最优路径
print("\n最优路径:")
for i, (state, action) in enumerate(optimal_path):
    if i == 0:
        print(f"起始点 ({state[0]+1},{state[1]+1}) -> {action}")
    elif action == "目标":
        print(f"到达目标 ({state[0]+1},{state[1]+1})")
    else:
        print(f"({optimal_path[i-1][0][0]+1},{optimal_path[i-1][0][1]+1}) -> {action} -> ({state[0]+1},{state[1]+1})")

# 展示最优策略网格
print("\n最优策略网格 (S=起点, G=目标, X=障碍物, .=未访问):")
for row in path_grid:
    print(' '.join(row))

# 最优策略网格 (S=起点, G=目标, X=障碍物, .=未访问):
# . → → G
# . S . X
# . . . .

价值函数:
[[7.23553439 8.06173439 8.97973439 9.99973439]
 [6.49195439 7.23553439 8.06173439 0.        ]
 [5.82273239 6.49195439 7.23553439 6.49195439]]
价值函数:
[[7.23553439 8.06173439 8.97973439 9.99973439]
 [6.49195439 7.23553439 8.06173439 0.        ]
 [5.82273239 6.49195439 7.23553439 6.49195439]]

最优路径:
起始点 (2,2) -> ↑
(2,2) -> → -> (1,2)
(1,2) -> → -> (1,3)
到达目标 (1,4)

最优策略网格 (S=起点, G=目标, X=障碍物, .=未访问):
. → → G
. S . X
. . . .
